# Chapter 10: Regularization & Cross-Validation Exercise

## Learning Objectives
After completing this exercise, you will be able to:
- Measure model performance reliably with k-fold cross-validation
- Recognize overfitting when the number of features grows
- Tune Ridge and Lasso regression with `GridSearchCV`
- Use Lasso to find out which features really matter
- Evaluate your final model honestly on a test set that was locked away

Let's put cross-validation and regularization to work on real medical data!

In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_validate, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Lasso can warn that it needs more iterations for very small alphas. The results are still
# usable, so we hide these warnings to keep the output readable.
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

## 10.1 Introduction & Motivation

In the notebook, we worked with a toy dataset with only one feature. Real datasets usually have many features. As soon as we add squared terms and interactions between features (like in Chapter 3), the number of features explodes. With 10 features, a degree 2 polynomial already has 65 features, and a degree 3 polynomial has 285!

More features means more flexibility, and more flexibility means a higher risk of overfitting. In this exercise, you will:
1. Find out how much you can trust a single train-test split
2. Use cross-validation to measure performance reliably
3. See overfitting appear when adding polynomial features
4. Use Ridge and Lasso to bring it back under control

## 10.2 Problem Setting

Diabetes is a chronic disease that affects hundreds of millions of people worldwide. For some patients, the disease gets worse quickly, while for others it remains stable for years. If doctors can predict early on which patients are at risk, they can start a more intensive treatment in time.

The dataset we will use today contains measurements of **442 diabetes patients**. For each patient, 10 measurements were taken at the start of the study (the "baseline"). One year later, the **progression** of the disease was measured as a number: the higher, the worse the disease has become. This is a famous dataset in statistics: it was used in the research paper that introduced one of the fast algorithms for computing the Lasso (Efron et al., 2004).

| Column | Meaning |
|---|---|
| `age` | Age in years |
| `sex` | Sex of the patient (coded as 1 or 2) |
| `bmi` | Body mass index |
| `bp` | Average blood pressure |
| `tc` | Total serum cholesterol |
| `ldl` | Low-density lipoproteins ("bad" cholesterol) |
| `hdl` | High-density lipoproteins ("good" cholesterol) |
| `tch` | Total cholesterol / HDL ratio |
| `ltg` | Log of the serum triglycerides level |
| `glu` | Blood sugar level |
| `progression` | **Target:** disease progression one year after baseline |

Our challenge: **Can we predict disease progression from the baseline measurements? And which measurements matter most?**

## 10.3 Model

Download the file "Diabetes_Progression.csv" from module 10 and put it in the same folder as this notebook.

In [ ]:
df = pd.read_csv("Diabetes_Progression.csv")
df.head()

##### Question 1: Explore the data. How many patients and features are there? Are there missing values? Plot a heatmap of the correlations. Which features are most strongly correlated with `progression`?

**Hints:**
- `df.shape`, `df.info()` and `df.describe()` give you a quick overview
- Use `sns.heatmap(df.corr(), cmap='RdBu_r', annot=True, fmt='.2f')` like in Chapter 3
- Look at the `progression` row of the heatmap

In [ ]:
# Your code here


**Your analysis:**

##### Question 2: Define the features `X` (all columns except `progression`) and the target `y`. Split the data into a training set (80%) and a test set (20%) with `random_state=42`. Train a `LinearRegression` model on the training set and calculate the $R^2$ and the MSE on both the training set and the test set.

**Reminder:** From now on, the test set is **locked away**. We only use it again at the very end (Question 8).

In [ ]:
# Your code here


**Your analysis:**

##### Question 3: How much can we trust the test score from Question 2? Repeat the train-test split and the training for `random_state` = 0, 1, ..., 19. Store the test $R^2$ of each run and plot them. What are the lowest and highest values? What does this tell you?

**Hints:**
- Use different variable names inside the loop (for example `X_tr`, `X_te`, ...) so you don't overwrite the split from Question 2
- `np.min()`, `np.max()` and `np.std()` summarize your results

In [ ]:
# Your code here


**Your analysis:**

##### Question 4: Use 10-fold cross-validation on the **training set** to evaluate the `LinearRegression` model. Use `KFold(n_splits=10, shuffle=True, random_state=42)` and `scoring='r2'`. Report the $R^2$ of each fold, the mean and the standard deviation. Compare the mean with your result from Question 2.

**Hint:** Store the `KFold` object in a variable called `kfold`, you will need it in the next questions.

In [ ]:
# Your code here


**Your analysis:**

## 10.4 Model Evaluation

Maybe the relationship is not purely linear? In Chapter 3 we improved our models by adding polynomial features. With several features, `PolynomialFeatures` also adds **interactions**: products of two features, like `bmi * bp`. Let's see if this helps.

##### Question 5: Build a pipeline with `PolynomialFeatures(degree=2, include_bias=False)`, `StandardScaler()` and `LinearRegression()`. How many features does the model use? Use `cross_validate` with `return_train_score=True` to calculate the mean train $R^2$ and the mean CV $R^2$. Repeat this for degree 3. Is the model overfitting?

**Hints:**
- `PolynomialFeatures(degree=2, include_bias=False).fit_transform(X_train).shape` shows the number of features
- `cross_validate(model, X_train, y_train, cv=kfold, scoring='r2', return_train_score=True)` returns a dictionary with the keys `'train_score'` and `'test_score'` (here, "test" means the validation fold)
- Compare with the train and CV $R^2$ of the plain linear model

In [ ]:
# Your code here


**Your analysis:**

##### Question 6: Replace `LinearRegression` by `Ridge` in the degree 2 pipeline. Use `GridSearchCV` (with `kfold` and `scoring='r2'`) to find the best `alpha` in `np.logspace(-3, 4, 29)`. Plot the mean CV $R^2$ against `alpha` and report the best `alpha` and its CV $R^2$. Did regularization fix the overfitting?

**Hints:**
- The parameter grid is `{'ridge__alpha': np.logspace(-3, 4, 29)}`
- After fitting, `grid.cv_results_['mean_test_score']` contains the mean CV $R^2$ for every `alpha`
- Use `plt.xscale('log')` for the x-axis

In [ ]:
# Your code here


**Your analysis:**

##### Question 7: Now do the same with `Lasso` (use `Lasso(max_iter=50000)`) and search `alpha` in `np.logspace(-2, 2, 17)`. Report the best `alpha` and its CV $R^2$. How many of the 65 coefficients does the best Lasso model set to zero? Which features does it keep? What does this tell a doctor?

**Hints:**
- The parameter grid is `{'lasso__alpha': np.logspace(-2, 2, 17)}`
- `grid.best_estimator_` is the best pipeline. Its last step (`[-1]`) is the Lasso model with the coefficients in `.coef_`
- The first step (`[0]`) is the `PolynomialFeatures` object. `.get_feature_names_out(X.columns)` gives the names of the 65 features
- This cell can take up to a minute to run

In [ ]:
# Your code here


**Your analysis:**

##### Question 8: It's time to open the locked test set! Evaluate the following four models on the test set: the plain linear model (Question 2), the degree 2 linear model without regularization, the best Ridge model (Question 6) and the best Lasso model (Question 7). Show the test $R^2$ and MSE in one table. Which model would you recommend to the hospital, and why?

**Hints:**
- The degree 2 model without regularization still needs to be trained on `X_train`
- Your `GridSearchCV` objects from Questions 6 and 7 were already retrained on the full training set (`refit`), so you can call `.predict()` on them directly
- Think about performance, but also about how easy each model is to explain to a doctor

In [ ]:
# Your code here


**Your analysis:**

## 10.5 Exercises

##### Question 1: See section 10.3
##### Question 2: See section 10.3
##### Question 3: See section 10.3
##### Question 4: See section 10.3
##### Question 5: See section 10.4
##### Question 6: See section 10.4
##### Question 7: See section 10.4
##### Question 8: See section 10.4

##### Question 9: Let's see Lasso's feature selection in action on the **original 10 features** (no polynomial features). Fit a pipeline with `StandardScaler()` and `Lasso(alpha=alpha)` on the training set for 50 values of `alpha` in `np.logspace(-1, 2, 50)`, and store the 10 coefficients each time. Plot each coefficient against `alpha` (one line per feature). This plot is called the **Lasso path**. In which order do the features disappear? Which two features are the last to be dropped?

**Hints:**
- Store the coefficients in a list, and turn it into a DataFrame with `pd.DataFrame(all_coefs, index=alphas, columns=X.columns)`
- `DataFrame.plot()` draws one line per column. Use `logx=True`

In [ ]:
# Your code here


**Your analysis:**

##### Question 10 (bonus): Ridge and Lasso can also be combined: the **Elastic Net** uses both penalties at the same time. Its parameter `l1_ratio` decides the mix (`l1_ratio=1` is pure Lasso, `l1_ratio=0` is pure Ridge). Use `GridSearchCV` with an `ElasticNet(max_iter=50000)` in the degree 2 pipeline, and search `alpha` in `[0.1, 0.3, 1, 3]` and `l1_ratio` in `[0.2, 0.5, 0.8]`. Does Elastic Net beat Ridge and Lasso?

**Hints:**
- `from sklearn.linear_model import ElasticNet`
- A parameter grid can contain several parameters: `GridSearchCV` will try every combination (here 4 × 3 = 12 combinations, each with 10 folds)

In [ ]:
# Your code here


**Your analysis:**